# 01 - Data and EDA
Run from the project folder. The first cell downloads and cleans the data if it is missing, so no other command is needed.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working folder:", os.getcwd())
from src.data import CLEAN, prepare
if not CLEAN.exists():
    print("no processed data yet, downloading and cleaning it now")
    prepare()
print("data ready:", CLEAN)

In [ ]:
import pandas as pd
from IPython.display import Image, display
from src.data import CLEAN, NUM, CAT
df = pd.read_csv(CLEAN)
print("rows, columns:", df.shape)
df.head()

## Rows per hospital file and class balance

In [ ]:
print(df.groupby("source").target.agg(rows="size", positives="sum"))
print()
print(df.target.value_counts().rename({0: "no disease", 1: "disease"}))

## Missing values
0 in chol and trestbps was already turned into missing during cleaning.

In [ ]:
miss = df.drop(columns="source").isna().sum()
pd.DataFrame({"missing": miss, "percent": (100 * miss / len(df)).round(1)}).query("missing > 0")

In [ ]:
print("missing per hospital file (%):")
(df.drop(columns="target").set_index("source").isna().groupby("source").mean() * 100).round(0)[["trestbps", "chol", "ca", "thal"]]

## Summary of numeric features

In [ ]:
df[NUM].describe().round(2)

## Plots (written to reports/figures by src/eda.py)

In [ ]:
from src.eda import main as run_eda
run_eda()

In [ ]:
for name in ["class_balance", "histograms", "correlation", "boxplots", "source_split"]:
    print(name)
    display(Image(f"reports/figures/{name}.png"))

## What I see
- Rows: Cleveland 303, Hungarian 294, Switzerland 123, VA 200.
- ca and thal have the most missing values, mostly from specific hospitals.
- Class balance and missing values differ by hospital, so I check per-hospital results in training.